In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import time

In [6]:
#df for feature sleection
import pandas as pd
df_select = pd.read_parquet(
    "train_preprocessed.parquet",
    engine="fastparquet",
)
df_select.shape

X_train_dt, y_train_dt = df_select, df_select['label']


In [ ]:
df_select = df_select.select_dtypes(include='number')
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ✓ STEP 1: Fill NaN with mean
mean_values = df_select.mean()
df_select_filled = df_select.fillna(mean_values)

# ✓ STEP 2: Use StandardScaler (better than manual standardization)
scaler = StandardScaler()
df_select_std = scaler.fit_transform(df_select_filled)

#applying PCA
#SVD set to randomized to avoid reading entire dataset rather focusing on the summary of the directions

start_time = time.time()

# Option 1: Keep 37 components of variance
# ✓ Use IncrementalPCA instead of PCA
# It processes data in batches (doesn't load everything into memory)
batch_size = 1000  # Adjust based on your RAM

ipca = IncrementalPCA(n_components=37, batch_size=batch_size)
X_train_pca = ipca.fit_transform(df_select_std)
end_time = time.time()
print(f"Time taken for PCA with 37 components: {end_time - start_time:.2f} seconds")  
print(f"Explained variance ratio for 37 components: {ipca.explained_variance_ratio_.sum():.4f}")      

In [ ]:
#Option 2 using decision Tree
print("METHOD 3: DECISION TREE FEATURE IMPORTANCE")
X_train_dt = DecisionTreeClassifier(random_state=42, max_depth=20)
X_train_dt.fit(df_select_std, df["label"])
    
feature_importance = pd.DataFrame({
        'feature': df_select.columns,
        'importance':X_train_dt.feature_importances_
 }).sort_values('importance', ascending=False)
threshold=np.percentile(feature_importance['importance'], 725)  # top 75% features
    
selected_features = feature_importance[
        feature_importance['importance'] > threshold
    ]['feature'].tolist()
